In [1]:
import polars as pl
import time

inicio = time.perf_counter()
df_polars = pl.read_parquet("../data/dataset_5M.parquet")
fin = time.perf_counter()

print("Tiempo de lectura:", fin - inicio, "segundos")

Tiempo de lectura: 0.19273799995426089 segundos


In [2]:
print("Dimensiones:", df_polars.shape)
print("\nEsquema de datos:")
print(df_polars.schema)

Dimensiones: (5000000, 5)

Esquema de datos:
Schema({'edad': Float64, 'ingresos': Float64, 'altura': Float64, 'experiencia': Int32, 'categoria': String})


In [3]:
inicio = time.perf_counter()

resumen_categoria = (
    df_polars
    .group_by("categoria")
    .agg([
        pl.len().alias("cantidad"),
        pl.col("edad").mean().alias("edad_media"),
        pl.col("ingresos").mean().alias("ingresos_media")
    ])
    .sort("categoria")
)

fin = time.perf_counter()
print("Tiempo de agregación:", fin - inicio, "segundos")

# Guardar resultado
resumen_categoria.write_csv("../resultados/tablas/polars_categoria.csv")
print("Tabla guardada en ../resultados/tablas/polars_categoria.csv")
resumen_categoria

Tiempo de agregación: 0.07162649999372661 segundos
Tabla guardada en ../resultados/tablas/polars_categoria.csv


categoria,cantidad,edad_media,ingresos_media
str,u32,f64,f64
"""A""",1000973,39.987784,24958.40589
"""B""",997803,40.027606,24973.816101
"""C""",999811,39.986273,24960.643637
"""D""",999884,39.992806,24959.606362
"""E""",1001529,39.994719,24945.888964


In [4]:
cuantiles_polars = df_polars.select([
    pl.col("edad").quantile(0.25).alias("p25"),
    pl.col("edad").quantile(0.50).alias("p50"),
    pl.col("edad").quantile(0.75).alias("p75")
])

cuantiles_polars.write_csv("../resultados/tablas/polars_cuantiles.csv")
print("Cuantiles guardados en ../resultados/tablas/polars_cuantiles.csv")
cuantiles_polars

Cuantiles guardados en ../resultados/tablas/polars_cuantiles.csv


p25,p50,p75
f64,f64,f64
31.898261,40.000474,48.091602


In [5]:
print("Tipos originales:")
print(df_polars.select(["edad", "ingresos", "altura"]).dtypes)

# Conversión explícita a Float32
df_float32 = df_polars.with_columns([
    pl.col("edad").cast(pl.Float32),
    pl.col("ingresos").cast(pl.Float32),
    pl.col("altura").cast(pl.Float32)
])

print("\nTipos convertidos a Float32:")
print(df_float32.select(["edad", "ingresos", "altura"]).dtypes)

Tipos originales:
[Float64, Float64, Float64]

Tipos convertidos a Float32:
[Float32, Float32, Float32]


In [6]:
memoria_original = df_polars.estimated_size(unit="mb")
memoria_float32 = df_float32.estimated_size(unit="mb")

print("Memoria original:", memoria_original, "MB")
print("Memoria Float32:", memoria_float32, "MB")

reduccion_memoria = (1 - memoria_float32 / memoria_original) * 100
print("Reducción de memoria:", reduccion_memoria, "%")

Memoria original: 138.28277587890625 MB
Memoria Float32: 81.06231689453125 MB
Reducción de memoria: 41.379310344827594 %


In [7]:
media_original = df_polars["edad"].mean()
media_float32 = df_float32["edad"].mean()

std_original = df_polars["edad"].std()
std_float32 = df_float32["edad"].std()

print("Media original:", media_original)
print("Media Float32:", media_float32)
print("STD original:", std_original)
print("STD Float32:", std_float32)

error_media = abs(media_original - media_float32) / abs(media_original) * 100
print("Diferencia relativa de la media:", error_media, "%")

Media original: 39.997821996896825
Media Float32: 39.99782199733484
STD original: 12.00137902522025
STD Float32: 12.001379013061523
Diferencia relativa de la media: 1.0950948016451908e-09 %


In [8]:
inicio = time.perf_counter()

df_lazy = (
    pl.scan_parquet("../data/dataset_5M.parquet")
    .group_by("categoria")
    .agg([
        pl.len().alias("cantidad"),
        pl.col("edad").mean().alias("edad_media"),
        pl.col("ingresos").mean().alias("ingresos_media")
    ])
    .sort("categoria")
    .collect()
)

fin = time.perf_counter()
print("Tiempo usando Lazy:", fin - inicio, "segundos")
df_lazy

Tiempo usando Lazy: 0.12343899998813868 segundos


categoria,cantidad,edad_media,ingresos_media
str,u32,f64,f64
"""A""",1000973,39.987784,24958.40589
"""B""",997803,40.027606,24973.816101
"""C""",999811,39.986273,24960.643637
"""D""",999884,39.992806,24959.606362
"""E""",1001529,39.994719,24945.888964
